# Laya training-data generator

Turns **real lectures with real ASR** into fine-tuning data for Laya (`verdict` + `boundary_status`).
Labels come from **ground truth** (Bin Baz transcripts), not from a model. A teacher model only confirms GT-silent cases,
and every disagreement goes to the review queue. Plan: `docs/plans/laya-datagen-plan.md`.

**Run top to bottom.** Everything expensive is cached in `data/laya_datagen/cache/` (HTML, audio, ASR, engine output,
teacher answers), so re-running a cell is cheap and editing an engine only re-runs detection.

Gates: **G1** (cell 9) audit precision ≥ 97 % before scaling · **G2** `laya.train.dry_run` skips 0 rows before training.

## 1. Config

In [1]:
import sys, os, json, collections
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "backend").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 200)
from backend.laya_datagen import config, cache
from backend.laya_datagen.engine_bridge import EngineBridge
from backend.laya_datagen.sources import binbaz
from backend.laya_datagen import asr as asr_mod, pipeline, review, reconcile, export, validate, benchmark

ENGINES        = ["v3", "v4"]            # engines to run on every lesson (add "v5" once it is registered in harness.js)
PRIMARY_ENGINE = "v4"                    # the engine whose envelopes are exported as training data
HADITH_CORPUS  = str(ROOT / "data/corpora/hadith_open_hadith_data.json")   # None -> hadiths inside data/corpus.json
ASR_BACKEND    = "groq"                  # "local" = CPU faster-whisper, ~0.85x realtime (CUDA is broken here)
ASR_MODEL      = "whisper-large-v3-turbo"
STATE_PROFILE  = "engine_agnostic"       # hides engine_score / engine_confidence_band from Laya (plan D2)
DATASET_VERSION = "v0"

# Bin Baz audio series (id, label). Pick series to fill gaps shown in cell 14.
SERIES = {
    "riyad_1":   ("https://binbaz.org.sa/audios/series/29/%D8%B1%D9%8A%D8%A7%D8%B6-%D8%A7%D9%84%D8%B5%D8%A7%D9%84%D8%AD%D9%8A%D9%86-%D8%A7%D9%84%D8%B4%D8%B1%D8%AD-%D8%A7%D9%84%D8%A7%D9%88%D9%84", "hadith + narrators"),
    "ibn_kathir":("https://binbaz.org.sa/audios/series/7/%D8%AA%D9%81%D8%B3%D9%8A%D8%B1-%D8%A7%D9%84%D9%82%D8%B1%D8%A7%D9%86-%D8%A7%D9%84%D8%B9%D8%B8%D9%8A%D9%85-%D8%A7%D8%A8%D9%86-%D9%83%D8%AB%D9%8A%D8%B1", "Quran-heavy"),
    "tawhid":    ("https://binbaz.org.sa/audios/series/83/%D8%B4%D8%B1%D8%AD-%D9%83%D8%AA%D8%A7%D8%A8-%D8%A7%D9%84%D8%AA%D9%88%D8%AD%D9%8A%D8%AF", "mixed Quran + hadith"),
    "wabil":     ("https://binbaz.org.sa/audios/series/42/%D8%A7%D9%84%D9%88%D8%A7%D8%A8%D9%84-%D8%A7%D9%84%D8%B5%D9%8A%D8%A8-%D9%85%D9%86-%D8%A7%D9%84%D9%83%D9%84%D9%85-%D8%A7%D9%84%D8%B7%D9%8A%D8%A8-%D8%A7%D9%84%D8%B4%D8%B1%D8%AD-%D8%A7%D9%84%D8%A7%D9%88%D9%84", "adhkar / dua"),
}
USE_SERIES = ["riyad_1"]
LESSONS_PER_SERIES = 5

## 2. Environment

In [2]:
import shutil, subprocess
for k in ("GROQ_API_KEY", "OPENROUTER_API_KEY", "GEMINI_API_KEY"):
    try: config.key(k); print("✔", k)
    except RuntimeError as e: print("✘", e)
print("node:", subprocess.run(["node", "-v"], capture_output=True, text=True).stdout.strip(), "| ffmpeg:", bool(shutil.which("ffmpeg")))

✔ GROQ_API_KEY
✔ OPENROUTER_API_KEY
✔ GEMINI_API_KEY
node: v24.22.1 | ffmpeg: True


## 3. Engine bridge — the app's real JS (v3 / v4), with parity checks

In [3]:
bridge = EngineBridge(hadith_path=HADITH_CORPUS); info = bridge.start()
print(info["quran"], "ayat |", info["hadiths"], "hadiths | fingerprint", bridge.fingerprint())
s = "قال الله تعالى وأما بنعمة ربك فحدث ثم قال الشيخ هذا كلام عادي وقال النبي صلى الله عليه وسلم إنما الأعمال بالنيات"
for e in ENGINES:
    print(e, [(d["sourceType"], d["displayRef"], d["confidenceBand"], d["matchedText"]) for d in bridge.detect(e, s.split())])

6236 ayat | 46891 hadiths | fingerprint c23b53f86cbdbe0b
v3 [('QURAN', 'الضحى: 11', 'VERIFIED', 'وأما بنعمة ربك فحدث'), ('HADITH', 'سنن أبي داود (2201)', 'CANDIDATE', 'إنما الأعمال بالنيات')]
v4 [('QURAN', 'الضحى: 11', 'VERIFIED', 'وأما بنعمة ربك فحدث')]


## 4. Source discovery and download

In [4]:
lessons = []
for name in USE_SERIES:
    urls = binbaz.series_items(SERIES[name][0], max_pages=2)[:LESSONS_PER_SERIES]
    for u in urls:
        l = binbaz.get_lesson(u); l.series = name; lessons.append(l)
df = pd.DataFrame([{"id": l.id, "title": l.title[:50], "blocks": len(l.blocks), "book": sum(b.kind == "book" for b in l.blocks),
                    "sheikh": sum(b.kind == "sheikh" for b in l.blocks), "mp3": bool(l.mp3)} for l in lessons]); df

,id,title,blocks,book,sheikh,mp3
0,binbaz_audio_2950,01 من: ( باب الإِخلاصِ وإحضار النيَّة في جميع الأع,89,33,56,True
1,binbaz_audio_561,02 من: ( باب التوبة ),60,28,32,True
2,binbaz_audio_565,03 من حديث (أن امرأة من جهينة أتت رسول الله ﷺ وهي,59,31,28,True
3,binbaz_audio_567,04 من حديث (إذا أراد الله بعبده خيرا عجل له العقوب,78,36,42,True
4,binbaz_audio_571,05 من حديث (إنكم لتعملون أعمالا هي أدق في أعينكم م,87,45,42,True


In [5]:
from tqdm.auto import tqdm
for l in tqdm(lessons, desc="mp3"): binbaz.download_mp3(l)

mp3:   0%|          | 0/5 [00:00<?, ?it/s]

## 5. ASR (cached)
Slow on this machine (CPU, ~0.85× realtime). Set `ASR_BACKEND = "groq"` once the Groq key is valid. Interrupt and re-run freely: finished lessons are cached.

In [6]:
asr_results = {}
for l in lessons:
    path = binbaz.download_mp3(l)
    print("ASR", l.id, end=" ... ")
    asr_results[l.id] = asr_mod.transcribe(path, backend=ASR_BACKEND, model=ASR_MODEL)
    print(len(asr_results[l.id]["words"]), "words")

ASR binbaz_audio_2950 ... 6719 words
ASR binbaz_audio_561 ... 6171 words
ASR binbaz_audio_565 ... 7477 words
ASR binbaz_audio_567 ... 7065 words
ASR binbaz_audio_571 ... 7139 words


## 6. GT parse + alignment + labels + case tags (doc gate)

In [7]:
processed = {}
for l in lessons:
    if l.id in asr_results:
        processed[l.id] = pipeline.process_cached(l, asr_results[l.id], bridge, ENGINES)
reports = pd.DataFrame([p["report"] for p in processed.values()]); reports

,doc_id,asr_tokens,gt_tokens,coverage_asr,coverage_gt,anchors,gt_quran_spans,gt_hadith_regions,unresolved_refs,passed_gate,by_tier
0,binbaz_audio_2950,6712,6690,0.798,0.796,2186,53,10,0,True,"{'neg_weak': 7, 'silent': 96, 'pos_strong': 77, 'unalignable': 3}"
1,binbaz_audio_561,6170,5791,0.734,0.778,2198,38,9,0,True,"{'unalignable': 27, 'pos_strong': 69, 'silent': 62, 'neg_weak': 1}"
2,binbaz_audio_565,7476,7053,0.767,0.811,2365,54,18,0,True,"{'silent': 67, 'pos_strong': 105, 'unalignable': 7, 'neg_weak': 4}"
3,binbaz_audio_567,7065,6699,0.760,0.799,2318,39,18,0,True,"{'pos_strong': 80, 'silent': 83, 'unalignable': 10}"
4,binbaz_audio_571,7112,6886,0.768,0.792,2078,60,17,1,True,"{'pos_strong': 98, 'silent': 73, 'unalignable': 5, 'neg_weak': 5, 'wrong_id'..."


In [8]:
records = [r for p in processed.values() for r in p["records"]]
df = pd.DataFrame([{"engine": r["engine"], "tier": r["label"]["tier"], "verdict": r["label"]["verdict"], "src": r["det"]["sourceType"],
                    "boundary": r["label"]["boundary"]} for r in records])
print(len(records), "candidate records")
pd.crosstab([df.engine, df.src], df.tier)

881 candidate records


tier           asr_mismatch  neg_weak  pos_strong  silent  unalignable  wrong_id
engine src                                                                      
v3     ADHKAR             0         0           1      25            9         0
       HADITH             0         0          72     151           16         0
       QURAN              1         3         122      11            9         1
v4     HADITH             0         0          90     187           12         0
       QURAN              0        14         144       7            6         0

## 7. Engine comparison (what each engine finds, scored against GT)

In [9]:
d = df[df.tier.isin(["pos_strong", "wrong_id", "neg_weak", "asr_mismatch"])]
cmp = d.groupby("engine").agg(n=("tier", "size"), correct=("verdict", lambda s: (s == "CORRECT").sum()), false=("verdict", lambda s: (s == "FALSE").sum()))
cmp["precision_vs_GT"] = (cmp.correct / cmp.n).round(3)
# recall: GT quran spans that no detection of this engine overlaps
rec = {}
for e in ENGINES:
    tot = hit = 0
    for l in lessons:
        if l.id not in processed or not processed[l.id]["records"]: continue
        ctx = pipeline.build_ctx(l, [w["word"] for w in asr_results[l.id]["words"]], bridge)
        found = [(r["det"]["wordStartIdx"], r["det"]["wordEndIdx"]) for r in processed[l.id]["records"] if r["engine"] == e and r["label"]["verdict"] == "CORRECT"]
        for s in ctx.gt.quran:
            a = [t for t in range(len(ctx.asr_tokens)) if ctx.al.asr2gt[t] >= 0 and s.gt_start <= ctx.al.asr2gt[t] < s.gt_end]
            if len(a) < 3: continue
            tot += 1; hit += any(ctx.word_tok[ws] <= a[0] < ctx.word_tok[we + 1] for ws, we in found)
    rec[e] = round(hit / max(1, tot), 3)
cmp["quran_recall_vs_GT"] = pd.Series(rec); cmp

,n,correct,false,precision_vs_GT,quran_recall_vs_GT
engine,,,,,
v3,200,195,5,0.975,0.520
v4,248,234,14,0.944,0.588


## 8. Gate G1 — audit 100 auto-labels
Mark each label right/wrong (and the boundary when shown). Needs **precision ≥ 97 %** on `pos_strong`/`wrong_id` and ≥ 90 % on boundaries.
Fix `gt_parse.py`/`align.py`/`label.py` (and bump `LABEL_VERSION`) if it fails — do not scale up first.

In [10]:
sample = review.audit_sample([r for r in records if r["engine"] == PRIMARY_ENGINE], n=100)
review.audit_widget(sample)

In [11]:
review.audit_results()

{}

## 9. Coverage matrix

In [12]:
prim = [r for r in records if r["engine"] == PRIMARY_ENGINE and r["label"]["tier"] in ("pos_strong", "wrong_id", "neg_weak")]
cov = collections.Counter((r["label"]["verdict"], r["det"]["sourceType"], c) for r in prim for c in r["cases"] if c.split(":")[0] in ("len", "noise", "flag", "band") or c in ("cue", "interrupted", "ref_ambiguity", "boundary_cut", "in_default_scope"))
pd.Series(cov).unstack(0).fillna(0).astype(int)

CORRECT  FALSE
HADITH band:AMBIGUOUS         37      0
       band:CANDIDATE          9      0
       band:VERIFIED          44      0
       cue                    43      0
       in_default_scope       90      0
       interrupted            11      0
       len:<=15               21      0
       len:<=7                 2      0
       len:>15                67      0
       noise:clean            51      0
       noise:mild             38      0
       noise:noisy             1      0
       ref_ambiguity          38      0
QURAN  band:AMBIGUOUS          1      2
       band:CANDIDATE         49     12
       band:VERIFIED          94      0
       boundary_cut           34      0
       cue                    77      1
       flag:ref_differs        3      0
       in_default_scope       50     14
       interrupted             3      0
       len:<=15               47      1
       len:<=3                 7      6
       len:<=7                70      7
       len:>15                20      0
       noise:clean            62     11
       noise:mild             50      2
       noise:noisy            32      1
       ref_ambiguity           5      4

## 10. Teachers
Cost estimate first; set `RUN_TEACHERS = True` to spend. Teacher answers are cached per envelope, so an engine edit only pays for new ones.
Teachers see the **original** envelope (with engine fields), exactly as in the app.

In [13]:
from backend.laya_datagen.teachers.jev import JevTeacher
from backend.laya_datagen.teachers.gemini import GeminiTeacher
from backend.laya_datagen.teachers.openrouter_chat import OpenRouterChatTeacher
from backend.laya_datagen.teachers.laya_local import LayaLocal

TEACHERS = {"jev": JevTeacher(), "gemini": GeminiTeacher("gemini-3.5-flash-lite")}   # add OpenRouterChatTeacher("deepseek/deepseek-chat") etc.
RUN_TEACHERS = False
BENCH_N = 300                      # GT-strong sample used to rank teachers
cands = [r for r in records if r["engine"] == PRIMARY_ENGINE]
todo = sum(1 for r in cands for t in TEACHERS.values() if t.cache.get(t.key(r["envelope"])) is None)
print(len(cands), "candidates;", todo, "uncached teacher calls (JEV ≈ $0.00005 each, Gemini free tier ≈ 12 RPM)")

460 candidates; 920 uncached teacher calls (JEV ≈ $0.00005 each, Gemini free tier ≈ 12 RPM)


In [14]:
teacher_out = {}
if RUN_TEACHERS:
    from tqdm.auto import tqdm
    for name, t in TEACHERS.items():
        bar = tqdm(total=len(cands), desc=name)
        teacher_out[name] = t.predict([r["envelope"] for r in cands], progress=lambda d, n: (setattr(bar, 'n', d), bar.refresh()))
        bar.close(); print(name, "cost $", round(t.cost_usd, 4))

## 11. Teacher benchmark (vs GT-strong labels; base Laya is the baseline to beat)

In [15]:
assert RUN_TEACHERS, "set RUN_TEACHERS = True in cell 10 first (this cell calls the teachers)"
strong = [i for i, r in enumerate(cands) if r["label"]["tier"] in ("pos_strong", "wrong_id")][:BENCH_N]
rows = {}
gold = [cands[i]["label"]["verdict"] for i in strong]; cases = [cands[i]["cases"] for i in strong]; bgold = [cands[i]["label"]["boundary"] for i in strong]
bench_t = dict(TEACHERS); bench_t["laya_base"] = LayaLocal()
for name, t in bench_t.items():
    preds = t.predict([cands[i]["envelope"] for i in strong])
    rows[name] = benchmark.score(gold, preds, cases, bgold); rows[name]["cost_usd"] = round(t.cost_usd, 4)
pd.DataFrame(rows).T

AssertionError: set RUN_TEACHERS = True in cell 10 first (this cell calls the teachers)

## 12. Reconcile + review queue

In [ ]:
stats = reconcile.reconcile(cands, teacher_out); print(stats)
queue = [r for r in cands if r.get("decision") == "review"]; print(len(queue), "in review queue")

In [ ]:
review.review_widget(queue)

## 13. Export, validation (gate G2)
Writes `data/laya_datagen/datasets/<version>/{train,val,test}.jsonl` (split by lecture) and runs `laya.train.dry_run` at the app's `max_len=1024 / head_max_len=256`.

In [ ]:
reconcile.reconcile(cands, teacher_out)
out = ROOT / "data/laya_datagen/datasets" / DATASET_VERSION
manifest = export.export_dataset(cands, out, profile=STATE_PROFILE, engine=PRIMARY_ENGINE,
                                 meta={"engines": ENGINES, "asr": [ASR_BACKEND, ASR_MODEL], "fingerprint": bridge.fingerprint(),
                                       "label_version": pipeline.LABEL_VERSION, "series": USE_SERIES})
print(json.dumps(manifest["counts"], indent=1))

In [ ]:
summary = validate.dry_run(str(out / "train.jsonl"))
assert not summary["skipped"], f"gate G2 failed: {summary['skipped']}"
trunc = validate.truncation_report(str(out / "train.jsonl"))
print(trunc)
assert trunc["share_over"] <= 0.02, "G2: >2% of rows are cut at max_len (the tail, i.e. boundary evidence, is lost) -> shorten context window or canonical text"
summary

## 14. Gap report — what to fetch next
Cells with few examples below, and which series usually fill them: tafsir Ibn Kathir → Quran-heavy · Al-Wabil al-Sayyib → adhkar/dua ·
Riyad / Bukhari / Bulugh → hadith + narrator names · Kitab al-Tawhid → mixed.

In [ ]:
tr = pd.read_json(out / "train.jsonl", lines=True)
print(len(tr), "train rows")
pd.DataFrame(collections.Counter((m["cases"][0], v) for m, v in zip(tr.meta, tr.gold.map(lambda g: max(g["verdict"]["probabilities"], key=g["verdict"]["probabilities"].get)))).most_common(20))